# 01 - Multi-Token Prediction draft와 검증

**학습 목표**: 여러 future token을 한 번에 제안하고 longest matching prefix만 확정하는 MTP 직관을 구현합니다.

**실행 방법**: Python 3/Jupyter에서 cell을 위에서 아래 순서로 실행합니다. 외부 패키지는 필요하지 않으며 Python 표준 기능만 사용합니다.

실제 GLM-OCR head가 아닌 toy reproduction입니다.

In [ ]:
# 문자열을 문자 token처럼 다뤄 외부 tokenizer 없이 acceptance 원리만 분리합니다.
def mtp_generate(target, width=5):
    output = []
    verifier_steps = 0
    accepted_lengths = []
    while len(output) < len(target):
        pos = len(output)
        draft = list(target[pos:pos + width])
        # 일부 block의 세 번째 후보를 틀리게 만들어 rejection을 관찰합니다.
        if (pos // width) % 2 == 1 and len(draft) > 2:
            draft[2] = '?'
        verifier_steps += 1
        accepted = 0
        for token, truth in zip(draft, target[pos:]):
            if token != truth:
                break
            output.append(token)
            accepted += 1
        if len(output) < len(target) and accepted < len(draft):
            # target verification이 알려 준 올바른 bonus token으로 최소 1칸 전진합니다.
            output.append(target[len(output)])
        accepted_lengths.append(accepted)
    return ''.join(output), verifier_steps, accepted_lengths

target = '<table><tr><td>42</td></tr></table>'
prediction, steps, accepted = mtp_generate(target, width=5)
print('prediction:', prediction)
print('AR steps:', len(target), 'MTP verifier steps:', steps)
print('accepted draft lengths:', accepted)
assert prediction == target
assert steps < len(target)


논문은 10-token prediction을 학습하고 실제 추론에서 평균 5.2 token/step을 보고합니다. width가 곧 acceptance는 아닙니다. 구조가 규칙적인 표 태그에서는 긴 prefix를 맞히기 쉽다는 가설을 확인하는 출발점입니다.